In [ ]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
# For NIfTI handling, nibabel is useful (pip install nibabel)
try:
    import nibabel as nib
except Exception:
    nib = None

## Data layout & NIfTI handling

Assumes `data/brain_mri/` with either `slices/` (png) or `niftis/` directory containing .nii/.nii.gz files. We'll show example code to extract mid-axial slices from NIfTI volumes.


In [ ]:
DATA_ROOT = Path('data/brain_mri')
SLICES_DIR = DATA_ROOT / 'slices'
NIFTI_DIR = DATA_ROOT / 'niftis'
print('Slices dir:', SLICES_DIR)
print('NIfTI dir:', NIFTI_DIR)

## Example: extract mid-axial slice from NIfTI

This cell requires `nibabel`. It extracts the central axial slice and saves it as PNG.


In [ ]:
if nib is None:
    print('nibabel not installed. Install with: pip install nibabel')
else:
    examples = list(NIFTI_DIR.glob('*.nii*'))[:5]
    for p in examples:
        vol = nib.load(str(p)).get_fdata()
        # Choose middle axial slice
        z = vol.shape[2] // 2
        slice = vol[:, :, z]
        slice = (slice - slice.min()) / (slice.max() - slice.min() + 1e-8)
        img = Image.fromarray((slice*255).astype('uint8'))
        outp = SLICES_DIR / (p.stem + '_slice.png')
        outp.parent.mkdir(parents=True, exist_ok=True)
        img.save(outp)
        print('Saved', outp)

## Transforms & `BrainMRIDataset`

Use transforms similar to other notebooks; ensure images are RGB for pretrained models by duplicating channel if grayscale.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
transform = T.Compose([
    T.Resize((256,256)),
    T.CenterCrop(224),
    T.Lambda(lambda img: img.convert('RGB') if img.mode!='RGB' else img),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BrainMRIDataset(Dataset):
    def __init__(self, root, slices_dir='slices', transform=None):
        self.root = Path(root)
        self.slices_dir = self.root / slices_dir
        self.transform = transform
        self.samples = []
        for p in sorted(self.slices_dir.glob('*')):
            if p.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                self.samples.append((str(p), 0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label

ds = BrainMRIDataset('data/brain_mri', transform=transform)
print('Samples:', len(ds))